# Régression linéaire par moindres carrés

Master Informatique Année 1, ISDIA MINF1, Aix-Marseille Université

Ce notebook suit l'énoncé : partie 1 avec `sklearn` (rapport), partie 2 avec l'implémentation maison (optionnelle).
Les cellules de texte marquées **À compléter** attendent vos observations une fois les cellules exécutées.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold

plt.rcParams['figure.figsize'] = (7, 4.5)

# 1. Régression linéaire par moindres carrés sous sklearn

## 1.1 Étude d'un régresseur linéaire simple

**Question 1.** Documentation : <http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html>

**Question 2.** Régression sur le tableau prix de vente / nombre de produits vendus, avec le paramétrage par défaut.
Rappel : `fit` attend une matrice `X` de forme `(n, 1)` et non un vecteur de forme `(n,)`, d'où le `reshape`.

In [ ]:
x = np.array([5.5, 6.0, 6.5, 6.0, 5.0, 6.5, 4.5, 5.0])
y = np.array([420, 380, 350, 400, 440, 380, 450, 420])

X = x.reshape(-1, 1)          # forme (8, 1)
print('Forme de x :', x.shape, '-> forme de X :', X.shape)

reg = LinearRegression()      # paramétrage par défaut (fit_intercept=True)
reg.fit(X, y)

y_pred = reg.predict(X)
mse = mean_squared_error(y, y_pred)
print('MSE sur l\'échantillon d\'apprentissage :', mse)

**Question 3.** Les attributs `coef_` et `intercept_`.

In [ ]:
print('coef_      :', reg.coef_)
print('intercept_ :', reg.intercept_)

`coef_` est le vecteur des pondérations $w_{1:d}$ associées aux variables explicatives (ici une seule valeur : la **pente** de la droite).
`intercept_` est le terme constant $w_0$ (l'**ordonnée à l'origine**, valeur prédite lorsque toutes les variables valent 0).
La droite apprise est donc $y = w_0 + w_1 x$ avec $w_0$ = `intercept_` et $w_1$ = `coef_[0]`.

**Question 4.** Le score (coefficient de détermination $R^2$) sur l'échantillon d'apprentissage.

In [ ]:
score = reg.score(X, y)
print('R^2 (apprentissage) :', score)

Le coefficient de détermination vaut
$$R^2 = 1 - \\frac{\\sum_i (y_i - \\hat y_i)^2}{\\sum_i (y_i - \\bar y)^2}.$$
Il mesure la part de la variance de $y$ expliquée par le modèle : 1 correspond à un ajustement parfait, 0 à un modèle qui prédit toujours la moyenne, et il peut être négatif si le modèle fait pire que la moyenne.

**À compléter** : commenter la valeur obtenue ci-dessus (une valeur proche de 1 indique un bon régresseur linéaire sur ces données, en gardant en tête que le score est calculé sur les données d'apprentissage et que l'échantillon ne compte que 8 points).

**Question 5.** Affichage de la droite apprise en rouge, superposée au nuage de points.

In [ ]:
xs = np.linspace(x.min() - 0.5, x.max() + 0.5, 100)
plt.scatter(x, y, label='données')
plt.plot(xs, reg.coef_[0] * xs + reg.intercept_, 'r', label='droite apprise')
plt.xlabel('prix de vente')
plt.ylabel('nombre de produits vendus')
plt.legend()
plt.title('Régression linéaire simple')
plt.show()

**Question 6.** Même apprentissage avec `fit_intercept=False`.

In [ ]:
reg0 = LinearRegression(fit_intercept=False)
reg0.fit(X, y)

print('coef_      :', reg0.coef_)
print('intercept_ :', reg0.intercept_)
print('MSE        :', mean_squared_error(y, reg0.predict(X)))
print('R^2        :', reg0.score(X, y))

plt.scatter(x, y, label='données')
plt.plot(xs, reg.coef_[0] * xs + reg.intercept_, 'r', label='avec intercept')
plt.plot(xs, reg0.coef_[0] * xs, 'g--', label='sans intercept')
plt.xlim(0, x.max() + 0.5)
plt.xlabel('prix de vente')
plt.ylabel('nombre de produits vendus')
plt.legend()
plt.show()

**Observation et explication.** Sans intercept, `intercept_` vaut 0 et la droite est contrainte de passer par l'origine : $y = w_1 x$.
Les données étant situées loin de l'origine (prix entre 4.5 et 6.5 pour des ventes autour de 400), avec une pente négative, cette contrainte impose une droite qui ne peut plus suivre la tendance : la MSE augmente fortement et le $R^2$ chute (il peut même devenir très négatif, car `score` reste calculé par rapport à la moyenne de $y$).
Le terme constant $w_0$ est donc indispensable dès que la relation ne passe pas naturellement par $(0, 0)$.

## 1.2 Jeu de données Eucalyptus

**Question 1.** Chargement des données dans deux tableaux numpy `h` (hauteur) et `c` (circonférence).
Le code cherche `eucalyptus.txt` puis `eucalypus.txt` (orthographe de l'énoncé) dans le dossier courant.
Si aucun fichier n'est trouvé, un jeu **synthétique** de même forme est créé uniquement pour que le notebook s'exécute : placez le vrai fichier à côté du notebook pour obtenir les vrais résultats.

In [ ]:
import os

fichier = None
for nom in ['eucalyptus.txt', 'eucalypus.txt']:
    if os.path.exists(nom):
        fichier = nom
        break

if fichier is not None:
    data = np.loadtxt(fichier)
    print('Fichier chargé :', fichier)
else:
    print('ATTENTION : fichier introuvable, utilisation de données synthétiques.')
    rng = np.random.default_rng(0)
    c_syn = rng.uniform(20, 80, 1000)
    h_syn = 10 + 8 * np.log(c_syn - 15) + rng.normal(0, 1.0, 1000)
    data = np.column_stack([h_syn, c_syn])

h = data[:, 0]                 # cible : hauteur
c = data[:, 1]                 # explication : circonférence
C = c.reshape(-1, 1)           # forme (n, 1) pour sklearn
print('n =', len(h), '| forme de C :', C.shape)

**Question 2.** Nuage de points.

In [ ]:
plt.scatter(c, h, s=8, alpha=0.5)
plt.xlabel('circonférence c')
plt.ylabel('hauteur h')
plt.title('Eucalyptus : hauteur en fonction de la circonférence')
plt.show()

print('Coefficient de corrélation de Pearson :', np.corrcoef(c, h)[0, 1])

**À compléter** : une corrélation linéaire semble-t-elle présente ? Justifier avec le nuage (les points s'alignent-ils autour d'une droite, ou plutôt autour d'une courbe concave ?) et avec le coefficient de corrélation de Pearson affiché ci-dessus.

**Question 3 (a).** Régresseur linéaire et droite apprise.

In [ ]:
reg_e = LinearRegression().fit(C, h)
print('w0 (intercept_) :', reg_e.intercept_)
print('w1 (coef_)      :', reg_e.coef_[0])

cs = np.linspace(c.min(), c.max(), 200)
plt.scatter(c, h, s=8, alpha=0.5, label='données')
plt.plot(cs, reg_e.predict(cs.reshape(-1, 1)), 'r', label='droite apprise')
plt.xlabel('circonférence c')
plt.ylabel('hauteur h')
plt.legend()
plt.show()

**Question 3 (b).** Estimation des performances par répétition de 10 hold-out.

**Choix de la répartition : 80 % apprentissage, 20 % test.**
Justification : avec plusieurs centaines d'exemples, 80 % suffisent pour que le modèle appris soit proche de celui appris sur toutes les données (biais d'estimation faible), et 20 % donnent un ensemble de test assez grand pour une estimation peu bruitée.
Répéter le découpage 10 fois avec des tirages aléatoires différents réduit la variance de l'estimation due à un unique découpage.

In [ ]:
def holdout_repete(F, h, n_rep=10, test_size=0.2, modele=LinearRegression):
    r2s, mses = [], []
    for seed in range(n_rep):
        F_tr, F_te, h_tr, h_te = train_test_split(F, h, test_size=test_size, random_state=seed)
        m = modele().fit(F_tr, h_tr)
        pred = m.predict(F_te)
        r2s.append(r2_score(h_te, pred))
        mses.append(mean_squared_error(h_te, pred))
    return np.array(r2s), np.array(mses)

r2_ho, mse_ho = holdout_repete(C, h)
print('Hold-out x10 (80/20)')
print(f'  R^2 : moyenne = {r2_ho.mean():.4f}, écart-type = {r2_ho.std():.4f}')
print(f'  MSE : moyenne = {mse_ho.mean():.4f}, écart-type = {mse_ho.std():.4f}')

**Question 3 (c).** Estimation par validation croisée avec `cross_val_score`.

Paramètres utiles :
- `estimator` : le modèle à évaluer (cloné et réappris à chaque pli) ;
- `X`, `y` : données et cible ;
- `cv` : nombre de plis (ou objet de découpage). On utilise 10 plis, avec `KFold(shuffle=True)` pour mélanger les exemples avant découpage (utile si le fichier est trié) ;
- `scoring` : métrique. Par défaut pour un régresseur c'est le $R^2$. Pour la MSE il faut passer `'neg_mean_squared_error'` : sklearn maximise toujours le score, donc la MSE est renvoyée **avec un signe moins**, qu'il faut inverser.

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=0)

def cv_scores(F, h, modele=None, cv=kf):
    modele = LinearRegression() if modele is None else modele
    r2 = cross_val_score(modele, F, h, cv=cv, scoring='r2')
    mse = -cross_val_score(modele, F, h, cv=cv, scoring='neg_mean_squared_error')
    return r2, mse

r2_cv, mse_cv = cv_scores(C, h)
print('Validation croisée (10 plis)')
print(f'  R^2 : moyenne = {r2_cv.mean():.4f}, écart-type = {r2_cv.std():.4f}')
print(f'  MSE : moyenne = {mse_cv.mean():.4f}, écart-type = {mse_cv.std():.4f}')

**Question 3 (d).** Comparaison des deux méthodes.

In [ ]:
print(f'{"méthode":<22}{"R2 moyen":>10}{"MSE moyenne":>14}')
print(f'{"hold-out 10 x (80/20)":<22}{r2_ho.mean():>10.4f}{mse_ho.mean():>14.4f}')
print(f'{"validation croisée 10":<22}{r2_cv.mean():>10.4f}{mse_cv.mean():>14.4f}')

**À compléter** : commenter les différences. Éléments d'analyse :
- Les deux méthodes donnent en général des valeurs proches ; les écarts viennent du hasard des découpages.
- Dans la validation croisée, chaque exemple sert exactement une fois en test et 9 fois en apprentissage : toutes les données sont exploitées, alors que les hold-out répétés peuvent tirer plusieurs fois le même exemple en test et jamais un autre.
- La validation croisée à 10 plis apprend sur 90 % des données (moins pessimiste) contre 80 % pour le hold-out ; les 10 hold-out ne sont pas indépendants entre eux, donc l'écart-type doit être interprété avec prudence.
- Coût : identique ici (10 apprentissages).

**Question 4.** Les résultats sont-ils satisfaisants ?

**À compléter** : comparer le $R^2$ obtenu ci-dessus à celui de l'exercice 1.1 (question 4) et à la valeur 1 d'un ajustement parfait. Un $R^2$ moyen (par exemple autour de 0.6) signifie qu'une partie importante de la variance de la hauteur reste inexpliquée par une droite : le modèle est correct mais perfectible, ce que le nuage de points (forme courbe) laisse déjà supposer.

**Question 5.** Hauteur prédite pour une circonférence de 22.8.

In [ ]:
h_pred = reg_e.predict(np.array([[22.8]]))
print('Hauteur prédite pour c = 22.8 :', h_pred[0])

**Question 6.** Modèle $f(c) = w_0 + w_1 c + w_2 \\log c$.

**(a)** Ajout de la colonne $\\log c$ : on passe à une régression multivariée avec $d = 2$.

In [ ]:
F2 = np.column_stack([c, np.log(c)])     # colonnes : c, log c
print('Forme de F2 :', F2.shape)

**(b)** Apprentissage, visualisation, estimation par validation croisée.

In [ ]:
reg_log = LinearRegression().fit(F2, h)
print('w0 :', reg_log.intercept_)
print('w1, w2 :', reg_log.coef_)

# Visualisation 3D : nuage (c, log c, h) et courbe apprise
cs = np.linspace(c.min(), c.max(), 200)
Fs = np.column_stack([cs, np.log(cs)])

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(c, np.log(c), h, s=6, alpha=0.4)
ax.plot(cs, np.log(cs), reg_log.predict(Fs), 'r', linewidth=2)
ax.set_xlabel('c')
ax.set_ylabel('log c')
ax.set_zlabel('h')
plt.show()

# Visualisation 2D : la courbe apprise dans le plan (c, h)
plt.scatter(c, h, s=8, alpha=0.5, label='données')
plt.plot(cs, reg_e.predict(cs.reshape(-1, 1)), 'r', label='c seul')
plt.plot(cs, reg_log.predict(Fs), 'g', label='c et log c')
plt.xlabel('circonférence c')
plt.ylabel('hauteur h')
plt.legend()
plt.show()

r2_log, mse_log = cv_scores(F2, h)
print(f'Validation croisée : R^2 = {r2_log.mean():.4f} | MSE = {mse_log.mean():.4f}')
print(f'Rappel modèle simple : R^2 = {r2_cv.mean():.4f} | MSE = {mse_cv.mean():.4f}')

**À compléter** : conclure. Si le $R^2$ augmente et la MSE diminue en validation croisée, le modèle avec $\\log c$ est meilleur : le terme logarithmique permet de suivre la courbure du nuage tout en restant un modèle **linéaire en les paramètres** $w$.

**(c)** Modèle $h = w_0 + w_1 c + w_2 \\log c + w_3 c^3$ (sans visualisation).

In [ ]:
F3 = np.column_stack([c, np.log(c), c ** 3])
r2_c3, mse_c3 = cv_scores(F3, h)
print(f'c, log c, c^3 : R^2 = {r2_c3.mean():.4f} | MSE = {mse_c3.mean():.4f}')

**(e)** Autres transformations de $c$, choisies d'après la forme du nuage.

Le nuage montre une croissance rapide de la hauteur pour les petites circonférences, puis un aplatissement : la courbe est **concave et croissante**. On teste donc des transformations concaves ($\\sqrt{c}$, $\\log c$, $1/c$ qui est croissante en $-1/c$ et sature) et, pour la souplesse, des termes polynomiaux ($c^2$).

In [ ]:
transformations = {
    'c'                       : lambda c: [c],
    'log c'                   : lambda c: [np.log(c)],
    'sqrt c'                  : lambda c: [np.sqrt(c)],
    '1/c'                     : lambda c: [1 / c],
    'c, log c'                : lambda c: [c, np.log(c)],
    'c, sqrt c'               : lambda c: [c, np.sqrt(c)],
    'c, c^2'                  : lambda c: [c, c ** 2],
    'c, 1/c'                  : lambda c: [c, 1 / c],
    'c, log c, c^3'           : lambda c: [c, np.log(c), c ** 3],
    'c, sqrt c, log c'        : lambda c: [c, np.sqrt(c), np.log(c)],
    'c, c^2, sqrt c, log c'   : lambda c: [c, c ** 2, np.sqrt(c), np.log(c)],
    'c, c^2, c^3, log c, 1/c' : lambda c: [c, c ** 2, c ** 3, np.log(c), 1 / c],
}

resultats = []
for nom, f in transformations.items():
    F = np.column_stack(f(c))
    r2, mse = cv_scores(F, h)
    resultats.append((nom, r2.mean(), r2.std(), mse.mean()))

resultats.sort(key=lambda t: -t[1])
print(f'{"variables":<28}{"R2 moyen":>10}{"ecart-type":>12}{"MSE moyenne":>14}')
for nom, m, s, e in resultats:
    print(f'{nom:<28}{m:>10.4f}{s:>12.4f}{e:>14.4f}')

**(d)** Meilleur modèle selon la validation croisée : celui qui figure en tête du tableau ci-dessus (plus grand $R^2$ moyen, plus petite MSE moyenne).

**À compléter** : nommer le meilleur modèle et commenter. Points à discuter :
- L'ajout de variables ne peut qu'améliorer le score **d'apprentissage** ; c'est pourquoi on compare par validation croisée.
- Au-delà d'un certain nombre de variables, le gain devient négligeable (voire négatif) : on privilégie le modèle le plus simple parmi ceux de performance comparable (risque de surapprentissage).
- Les transformations concaves (log, racine) collent à la forme du nuage et apportent l'essentiel du gain ; un $R^2$ qui plafonne indique que la circonférence seule n'explique pas toute la variance de la hauteur (bruit irréductible).

# 2. Programmation de la régression linéaire par moindres carrés (optionnel)

**Question 1.** Implémentation de l'algorithme :
1. ajouter une colonne de 1 à `X` ;
2. calculer $w = (X^\\top X)^{-1} X^\\top y$ ;
3. retourner $w$.

In [ ]:
def moindres_carres(X, y):
    """Renvoie w = (w0, w1, ..., wd) minimisant la somme des carrés des résidus."""
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    n = X.shape[0]
    Xb = np.ones([n, X.shape[1] + 1])
    Xb[:, 1:] = X                                   # première colonne de 1
    w = np.dot(np.dot(np.linalg.inv(np.dot(np.transpose(Xb), Xb)), np.transpose(Xb)), y)
    return w

def predire(X, w):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return w[0] + np.dot(X, w[1:])

**Question 2.** Test sur les données jouet de la section 1.1.

In [ ]:
w_toy = moindres_carres(x, y)
print('Programme maison : w0 =', w_toy[0], '| w1 =', w_toy[1])
print('sklearn          : w0 =', reg.intercept_, '| w1 =', reg.coef_[0])
print('Mêmes résultats ?', np.allclose([w_toy[0], w_toy[1]], [reg.intercept_, reg.coef_[0]]))

plt.scatter(x, y, label='données')
plt.plot(xs, w_toy[0] + w_toy[1] * xs, 'r', label='droite (programme maison)')
plt.plot(xs, reg.intercept_ + reg.coef_[0] * xs, 'k--', label='droite (sklearn)')
plt.legend()
plt.show()

**Question 3.** Test sur les données Eucalyptus initiales.

In [ ]:
w_e = moindres_carres(c, h)
print('Programme maison : w0 =', w_e[0], '| w1 =', w_e[1])
print('sklearn          : w0 =', reg_e.intercept_, '| w1 =', reg_e.coef_[0])
print('Mêmes résultats ?', np.allclose([w_e[0], w_e[1]], [reg_e.intercept_, reg_e.coef_[0]]))

**Question 4.** Score et MSE estimés par `train_test_split` avec le programme maison.

In [ ]:
C_tr, C_te, h_tr, h_te = train_test_split(c, h, test_size=0.2, random_state=0)

w_tr = moindres_carres(C_tr, h_tr)
pred_te = predire(C_te, w_tr)

mse_maison = np.mean((h_te - pred_te) ** 2)
r2_maison = 1 - np.sum((h_te - pred_te) ** 2) / np.sum((h_te - np.mean(h_te)) ** 2)

print('Programme maison : R^2 =', r2_maison, '| MSE =', mse_maison)

# Vérification avec sklearn sur le même découpage
m_sk = LinearRegression().fit(C_tr.reshape(-1, 1), h_tr)
print('sklearn          : R^2 =', m_sk.score(C_te.reshape(-1, 1), h_te),
      '| MSE =', mean_squared_error(h_te, m_sk.predict(C_te.reshape(-1, 1))))